# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
This dataset is described using a [Croissant schema](https://mlcommons.org/croissant/), provided via a remote JSON-LD file.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint
import warnings
warnings.filterwarnings('ignore')

# Define Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print top-level metadata
md = dataset.metadata
print(md.name)
print(md.description)
print(f"Published: {md.date_published}")

## 2. Data Overview
List available record sets and their field/column `@id`s, as defined in the Croissant schema.

In [ ]:
# List available record sets using their @id
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet: {rs['@id']}")
    print(f"  Name: {rs['name']}")
    print(f"  Description: {rs.get('description', '')}")
    print(f"  Fields and Columns:")
    for field in rs.get('field', []):
        print(f"    Field @id: {field['@id']}")
        print(f"      Name: {field.get('name', '')}")
        print(f"      DataType: {field.get('dataType', '')}")
        print(f"      Source column: {field.get('source', '')}")
    print('')

## 3. Data Extraction
Extract data from a selected record set using its `@id`. We'll choose one of the main record sets for demonstration based on the overview above.

In [ ]:
# Use the first record set as an example (update @id if preferred)
record_sets_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record set @id: {record_set_id}, Rows: {len(df)} Columns: {list(df.columns)}\n")

# For the rest of this notebook, focus on the first record set
main_record_set_id = record_sets_ids[0]
print(f"Using main record set: {main_record_set_id}")
print("Available columns:", dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's analyze and process the data: filter the records, normalize a numeric column, and group the data for summary statistics. We use columns and field `@id` (not display names).

In [ ]:
# Choose an appropriate numeric field for filtering/normalization
# Let's find one from the columns listed above, commonly 'mw' (molecular weight) or 'coverage_pct', etc.
numeric_fields = [c for c in dataframes[main_record_set_id].columns if dataframes[main_record_set_id][c].dtype in ['float64','int64']]
print(f"Numeric fields: {numeric_fields}")
numeric_field = None
if numeric_fields:
    numeric_field = numeric_fields[0]

if numeric_field is not None:
    threshold = dataframes[main_record_set_id][numeric_field].mean()
    filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field] > threshold].copy()
    print(f"Filtered {len(filtered_df)} records where {numeric_field} > mean ({threshold:.2f})")

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} sample:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by a likely categorical field, such as 'sample_id' or similar
    group_candidates = [c for c in dataframes[main_record_set_id].columns if dataframes[main_record_set_id][c].dtype == object and c != numeric_field]
    print(f"Possible group fields: {group_candidates}")
    group_field = group_candidates[0] if group_candidates else None
    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Let's plot a numeric field distribution and a boxplot grouped by a categorical field.

> _Note: If you are running in a headless environment (like Google Colab), use `%matplotlib inline` to show plots._

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if numeric_field is not None:
    plt.figure(figsize=(8,4))
    dataframes[main_record_set_id][numeric_field].hist(bins=30)
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.title(f"Distribution of {numeric_field}")
    plt.show()

    if group_field is not None:
        plt.figure(figsize=(10,5))
        # Only plot top 10 groups by count
        top_groups = filtered_df[group_field].value_counts().index[:10]
        filtered_df[filtered_df[group_field].isin(top_groups)].boxplot(column=numeric_field, by=group_field, rot=90)
        plt.title(f"{numeric_field} by {group_field} (top 10)")
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load a Croissant-described dataset with `mlcroissant`, explored available record sets and fields (referenced by their `@id`), extracted a main data table, performed basic filtering, normalization and grouping of numeric values, and visualized key distributions.

Researchers are encouraged to review field and column `@id` mappings to extend these analyses with biological domain knowledge and context.